# Methodology Spec v2 — Feature Extraction Pipeline
## `notebooks/11_METH_SPEC_v2_Features.ipynb`

**Spec:** `docs/METHODOLOGY_SPEC_v2.md` (v3.0)  
**Features:** F1 ATF · F2 TM · F4 D_eff · F5 Joint Gini  
**Output:** `results/meth_v2/feature_scalars.csv` + `results/meth_v2/run_metadata.json`

---

### Implementation cautions (§3.1 preamble)

| Topic | Guidance |
|-------|----------|
| **Branch D** | Deferred — implement only if Block 0 review shows persistent hand/foot marker loss. |
| **Block bootstrap** | Deferred (`run_bootstrap=False`). Implement as standalone post-hoc. |
| **`v2_viz_engine.py`** | Deferred — keep viz tidy-data-only after tidy exports are stable. |

In [1]:
# ── Block 0: Preamble ──────────────────────────────────────────────────

import sys, os, json, logging, datetime, re
from pathlib import Path

import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

PROJECT_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == "notebooks" else Path(os.getcwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.v2_feature_engine import (
    ALL_19_JOINTS, JOINT_GROUPS, TM_ENDPOINTS,
    apply_time_window, load_session,
    compute_quality_gates, validate_reference,
    compute_atf, compute_total_movement,
    build_pca_engine, compute_d_eff, compute_joint_gini,
    compute_ap_ratio, assemble_feature_row,
)

logging.basicConfig(level=logging.WARNING, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

DATA_ROOT = PROJECT_ROOT / "derivatives" / "step_06_kinematics"
RESULTS_DIR = PROJECT_ROOT / "results" / "meth_v2"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Data root    : {DATA_ROOT}")
print(f"Results dir  : {RESULTS_DIR}")

Project root : c:\Users\drorh\OneDrive - Mobileye\Desktop\gaga
Data root    : c:\Users\drorh\OneDrive - Mobileye\Desktop\gaga\derivatives\step_06_kinematics
Results dir  : c:\Users\drorh\OneDrive - Mobileye\Desktop\gaga\results\meth_v2


In [2]:
# ── Block 0.1: Parameter dicts ─────────────────────────────────────────

CONFIG = {
    "fs": 120.0,
    "artifact_critical_threshold": 0.30,
    "artifact_warning_threshold": 0.20,
    "per_joint_artifact_exclude": 0.30,
    "per_endpoint_artifact_flag": 0.20,
    "min_clean_fraction_pca": 0.70,
    "min_session_duration_s": 60.0,
    "ref_search_sec": 8.0,
    "static_search_step_sec": 0.1,
    "reference_variance_threshold": 100.0,
    "run_t2_isolation": False,
    "params_locked": False,
    "time_windows": {},
}

PARAMS_F1 = {
    "noise_floor_guard_mms": 1.0,
    "static_baseline_guard_mms": 50.0,
    "static_window_sec": 2.0,
    "artifact_warning_threshold": 0.20,
    "artifact_critical_threshold": 0.30,
    "noise_floor_override_mms_by_joint": {},
    "run_bootstrap": False,
}

PARAMS_F2 = {
    "min_segment_frames": 3,
    "normalize_by_duration": True,
    "run_bootstrap": False,
}

PARAMS_PCA_F4_F5 = {
    "kinematic_branch": "dynamics",
    "n_components": 19,
    "epsilon_deff": 1e-12,
    "min_clean_fraction": 0.70,
    "run_session_native_gini": True,
    "run_session_native_deff": True,
    "compute_ap_ratio": False,
    "run_bootstrap": False,
}

print("Parameter dicts defined: CONFIG, PARAMS_F1, PARAMS_F2, PARAMS_PCA_F4_F5")

Parameter dicts defined: CONFIG, PARAMS_F1, PARAMS_F2, PARAMS_PCA_F4_F5


In [3]:
# ── Block 0.2: Discover sessions & build registry ──────────────────────

parquet_files = sorted(DATA_ROOT.glob("*__kinematics_master.parquet"))

session_registry = []
for pf in parquet_files:
    name = pf.stem.replace("__kinematics_master", "")
    m = re.match(r"(\d+)_T(\d+)_P(\d+)_R(\d+)", name)
    if m:
        session_registry.append({
            "run_id": name,
            "subject_id": m.group(1),
            "timepoint": f"T{m.group(2)}",
            "protocol": f"P{m.group(3)}",
            "repetition": f"R{m.group(4)}",
            "path": str(pf),
        })
    else:
        session_registry.append({
            "run_id": name,
            "subject_id": name.split("_")[0],
            "timepoint": "unknown",
            "protocol": "unknown",
            "repetition": "unknown",
            "path": str(pf),
        })

registry_df = pd.DataFrame(session_registry)
all_subjects = sorted(registry_df["subject_id"].unique())

print(f"Found {len(parquet_files)} Parquet files, {len(all_subjects)} subjects: {all_subjects}")

Found 12 Parquet files, 2 subjects: ['651', '671']


---
## Section 1 — Subject & Session Selection

Use the controls below to:
1. **Pick a subject** to analyze
2. **Select / deselect individual sessions** (checkboxes)
3. **Choose the reference session** for PCA anchoring (T1 default)
4. **Set time-window crops** per session (optional — leave 0 / 0 to use full recording)

In [4]:
# ── Section 1: Interactive session selector ────────────────────────────

# ── Shared mutable state ──────────────────────────────────────────────
_UI = {
    "session_checkboxes": {},
    "crop_widgets": {},
    "ref_dropdown": None,
}

# ── Subject dropdown ──────────────────────────────────────────────────
w_subject = widgets.Dropdown(
    options=[(f"Subject {s}", s) for s in all_subjects],
    value=all_subjects[0],
    description="Subject:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="280px"),
)

# ── Container for the dynamic session panel ──────────────────────────
w_session_panel = widgets.Output()
w_quality_panel = widgets.Output()


def _artifact_badge(frac):
    pct = frac * 100
    if pct >= 30:
        color, label = "#d9534f", "CRITICAL"
    elif pct >= 20:
        color, label = "#f0ad4e", "WARNING"
    else:
        color, label = "#5cb85c", "OK"
    return f'<span style="background:{color};color:#fff;padding:1px 6px;border-radius:3px;font-size:0.85em">{label} {pct:.1f}%</span>'


def _short_id(run_id, max_len=55):
    return run_id if len(run_id) <= max_len else run_id[:max_len] + "..."


def _build_session_panel(subject_id):
    """Rebuild the session selection panel with artifact & duration stats."""
    sub_df = registry_df[registry_df["subject_id"] == subject_id].copy()
    sub_df = sub_df.sort_values(["timepoint", "repetition"]).reset_index(drop=True)

    _UI["session_checkboxes"] = {}
    _UI["crop_widgets"] = {}

    # Pre-scan each session file for quick stats
    session_stats = {}
    for _, r in sub_df.iterrows():
        try:
            sdf = pd.read_parquet(r["path"])
            n_frames = len(sdf)
            dur_s = n_frames / CONFIG["fs"]
            art_cols = [f"{j}__is_artifact" for j in ALL_19_JOINTS if f"{j}__is_artifact" in sdf.columns]
            if art_cols and n_frames > 0:
                global_art = float(sdf[art_cols].any(axis=1).sum()) / n_frames
            else:
                global_art = 0.0
            session_stats[r["run_id"]] = {"n_frames": n_frames, "dur_s": dur_s, "global_art": global_art}
        except Exception:
            session_stats[r["run_id"]] = {"n_frames": 0, "dur_s": 0.0, "global_art": 1.0}

    rows = []
    for _, r in sub_df.iterrows():
        rid = r["run_id"]
        st = session_stats[rid]

        cb = widgets.Checkbox(
            value=True,
            description="",
            indent=False,
            layout=widgets.Layout(width="30px"),
        )
        _UI["session_checkboxes"][rid] = cb

        badge = _artifact_badge(st["global_art"])
        lbl = widgets.HTML(
            value=(
                f"<b>{r['timepoint']} {r['repetition']}</b> &nbsp; "
                f"<span style='color:#444'>{st['n_frames']:,} fr</span> &nbsp; "
                f"<span style='color:#444'>{st['dur_s']:.1f}s</span> &nbsp; "
                f"{badge} &nbsp; "
                f"<span style='color:#888;font-size:0.8em'>{_short_id(rid, 45)}</span>"
            ),
            layout=widgets.Layout(width="700px"),
        )

        crop_start = widgets.FloatText(
            value=0.0, description="from:",
            style={"description_width": "38px"},
            layout=widgets.Layout(width="120px"),
        )
        crop_end = widgets.FloatText(
            value=0.0, description="to:",
            style={"description_width": "25px"},
            layout=widgets.Layout(width="110px"),
        )
        _UI["crop_widgets"][rid] = (crop_start, crop_end)

        row = widgets.HBox([cb, lbl, crop_start, crop_end])
        rows.append(row)

    # Reference dropdown
    ref_options = [(f"{r['timepoint']} {r['repetition']}  ({_short_id(r['run_id'], 40)})", r["run_id"])
                   for _, r in sub_df.iterrows()]
    t1_default = sub_df[sub_df["timepoint"] == "T1"].iloc[0]["run_id"] if len(sub_df[sub_df["timepoint"] == "T1"]) > 0 else sub_df.iloc[0]["run_id"]

    w_ref = widgets.Dropdown(
        options=ref_options,
        value=t1_default,
        description="Reference (PCA anchor):",
        style={"description_width": "160px"},
        layout=widgets.Layout(width="700px"),
    )
    _UI["ref_dropdown"] = w_ref

    header = widgets.HTML(
        "<div style='margin:4px 0 6px 0;font-size:0.9em'>"
        "<b style='width:30px;display:inline-block'>&#10003;</b>"
        "<b>Session &nbsp; Frames &nbsp; Duration &nbsp; Artifact</b>"
        "<span style='margin-left:260px'><b>Time crop (s)</b> &mdash; 0/0 = full</span>"
        "</div>"
    )

    with w_session_panel:
        clear_output(wait=True)
        display(widgets.VBox(
            [widgets.HTML(f"<h4>Sessions for Subject {subject_id}</h4>"),
             header] + rows +
            [widgets.HTML("<hr style='margin:8px 0'>"), w_ref]
        ))


def _on_subject_change(change):
    _build_session_panel(change["new"])
    with w_quality_panel:
        clear_output()


w_subject.observe(_on_subject_change, names="value")
_build_session_panel(w_subject.value)

display(widgets.VBox([
    widgets.HTML("<h3>Section 1 — Subject & Session Configuration</h3>"),
    w_subject,
    w_session_panel,
]))

In [17]:
# ── Block 0.3: Preview quality & load ──────────────────────────────────
# Click this button to scan quality gates *before* committing to the pipeline.

w_preview_btn = widgets.Button(
    description="  Preview Session Quality",
    button_style="info",
    icon="table",
    layout=widgets.Layout(width="260px", height="36px"),
)


def _on_preview(b):
    with w_quality_panel:
        clear_output(wait=True)
        display(HTML("<i>Loading sessions & computing quality gates...</i>"))

        subject_id = w_subject.value
        sub_df = registry_df[registry_df["subject_id"] == subject_id]

        selected_ids = [
            rid for rid, cb in _UI["session_checkboxes"].items() if cb.value
        ]
        if not selected_ids:
            clear_output(wait=True)
            display(HTML("<b style='color:red'>No sessions selected.</b>"))
            return

        # Load selected sessions
        _sessions = {}
        for rid in selected_ids:
            row = sub_df[sub_df["run_id"] == rid].iloc[0]
            df = load_session(row["path"])
            # Apply crop if set
            crop = _UI["crop_widgets"].get(rid)
            if crop:
                cs, ce = crop[0].value, crop[1].value
                if cs > 0 or ce > 0:
                    t_start = cs if cs > 0 else None
                    t_end = ce if ce > 0 else None
                    df = apply_time_window(df, "time_s", t_start, t_end)
            _sessions[rid] = df

        qdf = compute_quality_gates(_sessions, CONFIG)

        # Build HTML table
        rows_html = ""
        for _, qr in qdf.iterrows():
            rid = qr["run_id"]
            reg = sub_df[sub_df["run_id"] == rid].iloc[0]
            badge = _artifact_badge(qr["global_artifact_frac"])
            pca_style = 'color:red;font-weight:bold' if qr['pca_low_confidence'] else ''
            dur_style = 'color:orange;font-weight:bold' if qr['short_session'] else ''
            rows_html += (
                f"<tr>"
                f"<td>{reg['timepoint']} {reg['repetition']}</td>"
                f"<td style='text-align:right'>{qr['n_frames']:,}</td>"
                f"<td style='text-align:right;{dur_style}'>{qr['duration_s']:.1f}s</td>"
                f"<td>{badge}</td>"
                f"<td style='text-align:right;{pca_style}'>{qr['clean_fraction_pca']:.1%}</td>"
                f"<td style='text-align:right'>{qr['clean_duration_s']:.1f}s</td>"
                f"<td>{'<b style=color:red>EXCLUDE</b>' if qr['hard_exclude'] else '<span style=color:green>OK</span>'}</td>"
                f"<td style='font-size:0.8em;color:#555'>{_short_id(rid, 50)}</td>"
                f"</tr>"
            )

        table_html = (
            "<table style='border-collapse:collapse;font-family:monospace;font-size:0.9em;margin-top:8px'>"
            "<thead><tr style='background:#2c3e50;color:#fff'>"
            "<th style='padding:4px 8px'>Session</th>"
            "<th style='padding:4px 8px'>Frames</th>"
            "<th style='padding:4px 8px'>Duration</th>"
            "<th style='padding:4px 8px'>Artifact</th>"
            "<th style='padding:4px 8px'>PCA Clean</th>"
            "<th style='padding:4px 8px'>Clean Dur</th>"
            "<th style='padding:4px 8px'>Gate</th>"
            "<th style='padding:4px 8px'>Run ID</th>"
            "</tr></thead>"
            f"<tbody>{rows_html}</tbody></table>"
        )

        # Validate reference
        ref_id = _UI["ref_dropdown"].value
        ref_valid, ref_msg = validate_reference(qdf, ref_id)
        ref_color = "green" if ref_valid else "red"

        clear_output(wait=True)
        display(HTML(
            f"<b>{len(selected_ids)} sessions loaded</b>"
            f"{table_html}"
            f"<p style='margin-top:8px'>"
            f"<b>Reference:</b> {_short_id(ref_id, 55)} — "
            f"<span style='color:{ref_color}'>{ref_msg}</span></p>"
        ))


w_preview_btn.on_click(_on_preview)

display(widgets.VBox([
    w_preview_btn,
    w_quality_panel,
]))

In [18]:
# ── Block 0.4: Commit selections → load data ──────────────────────────
# This cell reads the widget state and populates the pipeline variables.
# Run this cell after you are satisfied with the preview above.

SUBJECT_ID = w_subject.value
REFERENCE_RUN_ID = _UI["ref_dropdown"].value

# Collect selected session IDs
active_run_ids = [
    rid for rid, cb in _UI["session_checkboxes"].items() if cb.value
]

# Load sessions with optional time crops
active_sessions: dict[str, pd.DataFrame] = {}
sub_df = registry_df[registry_df["subject_id"] == SUBJECT_ID]

for rid in active_run_ids:
    row = sub_df[sub_df["run_id"] == rid].iloc[0]
    df = load_session(row["path"])
    crop = _UI["crop_widgets"].get(rid)
    if crop:
        cs, ce = crop[0].value, crop[1].value
        if cs > 0 or ce > 0:
            t_start = cs if cs > 0 else None
            t_end = ce if ce > 0 else None
            df = apply_time_window(df, "time_s", t_start, t_end)
            CONFIG["time_windows"][rid] = {"start": cs, "end": ce}
    active_sessions[rid] = df

# Quality gates
quality_df = compute_quality_gates(active_sessions, CONFIG)

# Auto-exclude hard failures (unless manually forced)
auto_excluded = []
for _, qr in quality_df.iterrows():
    if qr["hard_exclude"] and qr["run_id"] != REFERENCE_RUN_ID:
        auto_excluded.append(qr["run_id"])

for rid in auto_excluded:
    if rid in active_sessions:
        del active_sessions[rid]
        active_run_ids.remove(rid)

# Validate reference
is_valid, msg = validate_reference(quality_df, REFERENCE_RUN_ID)

print(f"Subject:    {SUBJECT_ID}")
print(f"Reference:  {REFERENCE_RUN_ID}")
print(f"  Valid:    {msg}")
print(f"Active:     {len(active_run_ids)} sessions")
for rid in active_run_ids:
    n = len(active_sessions[rid])
    dur = n / CONFIG['fs']
    tp = sub_df[sub_df['run_id'] == rid].iloc[0]['timepoint']
    rep = sub_df[sub_df['run_id'] == rid].iloc[0]['repetition']
    marker = " ← REFERENCE" if rid == REFERENCE_RUN_ID else ""
    print(f"  {tp} {rep}: {n:,} frames ({dur:.1f}s){marker}")
if auto_excluded:
    print(f"Auto-excluded (artifact > 30%): {auto_excluded}")

if not is_valid:
    raise RuntimeError(f"PIPELINE HALT — {msg}")

Subject:    671
Reference:  671_T1_P2_R1_Take 2026-01-06 03.57.12 PM_002
  Valid:    Reference session passed all validation gates.
Active:     3 sessions
  T1 R1: 13,201 frames (110.0s) ← REFERENCE
  T2 R1: 13,201 frames (110.0s)
  T3 R1: 13,201 frames (110.0s)


---
## Block 1 — F1: Active Time Fraction (ATF)

Per-joint threshold-crossing rate using session-adaptive noise floor from `src/pulsicity.py`.  
**Math:** $\text{ATF}_{i,s} = \frac{\sum \mathbb{1}[v_i(t) > V_{i,s} \wedge a_i(t)=0]}{\sum \mathbb{1}[a_i(t)=0]}$  
**Whole-body:** $\text{ATF}_{wb} = \text{median}_{i \in \{1..19\}}(\text{ATF}_{i,s})$

In [19]:
# ── Block 1: F1 ATF ───────────────────────────────────────────────────

atf_results: dict[str, object] = {}

for rid in active_run_ids:
    atf_results[rid] = compute_atf(active_sessions[rid], rid, PARAMS_F1, CONFIG)
    r = atf_results[rid]
    print(f"  {rid[:50]}: ATF_wb={r.atf_wb:.4f}  axial={r.atf_axial:.4f}  periph={r.atf_peripheral:.4f}")

# Noise-floor audit table
nf_rows = []
for rid, res in atf_results.items():
    for j in ALL_19_JOINTS:
        nf = res.noise_floors.get(j, {})
        nf_rows.append({
            "run_id": rid,
            "joint": j,
            "noise_floor_mms": nf.get("V", np.nan),
            "method": nf.get("noise_floor_source", "unknown"),
            "low_confidence": nf.get("noise_floor_low_confidence", False),
            "atf": res.atf_per_joint.get(j, np.nan),
        })

nf_audit = pd.DataFrame(nf_rows)
print(f"\nNoise-floor audit: {len(nf_audit)} rows")
display(nf_audit.pivot_table(index="joint", columns="run_id", values="atf", aggfunc="first").round(4))

WARNING | compute_noise_floor(Spine): static window mean 50.6 mm/s >= guard 50.0 mm/s. Using 5th-percentile fallback V=17.19 mm/s.
WARNING | compute_noise_floor(Spine1): static window mean 212.0 mm/s >= guard 50.0 mm/s. Using 5th-percentile fallback V=62.77 mm/s.
WARNING | compute_noise_floor(Neck): static window mean 450.5 mm/s >= guard 50.0 mm/s. Using 5th-percentile fallback V=117.29 mm/s.
WARNING | compute_noise_floor(Head): static window mean 596.4 mm/s >= guard 50.0 mm/s. Using 5th-percentile fallback V=162.45 mm/s.
WARNING | compute_noise_floor(LeftShoulder): static window mean 391.5 mm/s >= guard 50.0 mm/s. Using 5th-percentile fallback V=107.11 mm/s.
WARNING | compute_noise_floor(LeftArm): static window mean 439.4 mm/s >= guard 50.0 mm/s. Using 5th-percentile fallback V=171.45 mm/s.
WARNING | compute_noise_floor(LeftForeArm): static window mean 860.5 mm/s >= guard 50.0 mm/s. Using 5th-percentile fallback V=447.61 mm/s.
WARNING | compute_noise_floor(LeftHand): static window mea

  671_T1_P2_R1_Take 2026-01-06 03.57.12 PM_002: ATF_wb=0.9499  axial=0.9499  periph=0.9500


WARNING | compute_noise_floor(RightUpLeg): static window mean 135.3 mm/s >= guard 50.0 mm/s. Using 5th-percentile fallback V=35.49 mm/s.
WARNING | compute_noise_floor(RightLeg): static window mean 509.5 mm/s >= guard 50.0 mm/s. Using 5th-percentile fallback V=131.81 mm/s.
WARNING | compute_noise_floor(RightFoot): static window mean 562.2 mm/s >= guard 50.0 mm/s. Using 5th-percentile fallback V=150.50 mm/s.
WARNING | compute_noise_floor(Spine1): static window mean 154.5 mm/s >= guard 50.0 mm/s. Using 5th-percentile fallback V=160.49 mm/s.
WARNING | compute_noise_floor(Neck): static window mean 224.1 mm/s >= guard 50.0 mm/s. Using 5th-percentile fallback V=213.09 mm/s.
WARNING | compute_noise_floor(Head): static window mean 353.5 mm/s >= guard 50.0 mm/s. Using 5th-percentile fallback V=274.92 mm/s.
WARNING | compute_noise_floor(LeftShoulder): static window mean 247.7 mm/s >= guard 50.0 mm/s. Using 5th-percentile fallback V=211.82 mm/s.
WARNING | compute_noise_floor(LeftArm): static windo

  671_T2_P2_R1_Take 2026-01-15 04.35.25 PM_006: ATF_wb=0.9499  axial=0.9499  periph=0.9500


WARNING | compute_noise_floor(RightLeg): static window mean 267.7 mm/s >= guard 50.0 mm/s. Using 5th-percentile fallback V=148.14 mm/s.
WARNING | compute_noise_floor(RightFoot): static window mean 187.9 mm/s >= guard 50.0 mm/s. Using 5th-percentile fallback V=161.04 mm/s.


  671_T3_P2_R1_Take 2026-02-03 08.05.01 PM_001: ATF_wb=0.9499  axial=0.9499  periph=0.9500

Noise-floor audit: 57 rows


run_id,671_T1_P2_R1_Take 2026-01-06 03.57.12 PM_002,671_T2_P2_R1_Take 2026-01-15 04.35.25 PM_006,671_T3_P2_R1_Take 2026-02-03 08.05.01 PM_001
joint,,,
Head,0.9499,0.9499,0.9499
Hips,0.0000,0.0000,0.0000
LeftArm,0.9499,0.9499,0.9499
LeftFoot,0.9499,0.9500,0.9500
LeftForeArm,0.9500,0.9500,0.9500
LeftHand,0.9500,0.9500,0.9499
LeftLeg,0.9500,0.9500,0.9499
LeftShoulder,0.9499,0.9499,0.9499
LeftUpLeg,0.9499,0.9499,0.9499


In [ ]:
# ── Block 1.1: Noise-floor override panel ─────────────────────────────
# Review the audit table above. If any joint shows a low-confidence or
# implausible noise floor, enter an override value here then re-run Block 1.

_low_conf_joints = sorted(nf_audit[nf_audit["low_confidence"]]["joint"].unique())

w_nf_overrides: dict[str, widgets.FloatText] = {}
nf_rows_ui = []

for j in ALL_19_JOINTS:
    # Current median noise floor across sessions
    med_nf = nf_audit[nf_audit["joint"] == j]["noise_floor_mms"].median()
    is_flagged = j in _low_conf_joints

    flag_html = (
        " <span style='color:#d9534f;font-weight:bold'>&#9888; low-confidence</span>"
        if is_flagged else ""
    )
    lbl = widgets.HTML(
        value=(
            f"<span style='font-family:monospace;width:260px;display:inline-block'>"
            f"<b>{j}</b>{flag_html}</span>"
            f"<span style='color:#666;font-size:0.85em'>  median V = {med_nf:.2f} mm/s</span>"
        ),
        layout=widgets.Layout(width="480px"),
    )

    override_input = widgets.FloatText(
        value=0.0,
        description="override:",
        style={"description_width": "60px"},
        layout=widgets.Layout(width="170px"),
        disabled=False,
    )
    w_nf_overrides[j] = override_input
    nf_rows_ui.append(widgets.HBox([lbl, override_input]))

w_nf_apply_btn = widgets.Button(
    description="  Apply Overrides & Re-run F1",
    button_style="warning",
    icon="refresh",
    layout=widgets.Layout(width="280px", height="36px"),
)
w_nf_status = widgets.Output()


def _on_nf_apply(b):
    """Push overrides into PARAMS_F1 and re-compute ATF."""
    overrides = {}
    for j, w in w_nf_overrides.items():
        if w.value > 0:
            overrides[j] = w.value
    PARAMS_F1["noise_floor_override_mms_by_joint"] = overrides

    with w_nf_status:
        clear_output(wait=True)
        if overrides:
            display(HTML(
                f"<b>Overrides applied for {len(overrides)} joint(s):</b> "
                + ", ".join(f"{j} = {v:.1f}" for j, v in overrides.items())
                + "<br><i>Re-run <b>Block 1</b> above to recompute ATF with these overrides.</i>"
            ))
        else:
            display(HTML("<i>No overrides set (all zeros). PARAMS_F1 cleared.</i>"))


w_nf_apply_btn.on_click(_on_nf_apply)

if _low_conf_joints:
    intro = (
        f"<p style='color:#d9534f;font-weight:bold'>"
        f"&#9888; {len(_low_conf_joints)} joint(s) flagged low-confidence: "
        f"{', '.join(_low_conf_joints)}</p>"
    )
else:
    intro = "<p style='color:#5cb85c'>All joints have confident noise floors.</p>"

display(widgets.VBox(
    [widgets.HTML("<h4>Noise-Floor Overrides</h4>"),
     widgets.HTML(intro),
     widgets.HTML(
         "<p style='font-size:0.85em;color:#666'>"
         "Enter a positive mm/s value to override a joint's noise floor. "
         "Leave at 0 to keep the auto-computed value. "
         "Click Apply, then re-run Block 1.</p>"
     )]
    + nf_rows_ui
    + [widgets.HTML("<hr style='margin:6px 0'>"),
       w_nf_apply_btn,
       w_nf_status]
))

---
## Block 2 — F2: Total Movement (TM)

Cumulative endpoint path length (4 endpoints) using contiguous-run logic.  
**Math:** $\text{TM}_s = \sum_t \sum_e \|\mathbf{p}_e(t) - \mathbf{p}_e(t-1)\|_2$ (clean frame pairs only)  
**Rate:** $\text{TM}_{\text{rate}} = \text{TM}_s / \text{clean\_duration\_tm\_s}$

In [21]:
# ── Block 2: F2 TM ────────────────────────────────────────────────────

tm_results: dict[str, object] = {}

for rid in active_run_ids:
    tm_results[rid] = compute_total_movement(active_sessions[rid], rid, PARAMS_F2, CONFIG)
    r = tm_results[rid]
    print(
        f"  {rid[:50]}: TM={r.tm_total_mm/1e6:.2f} km  "
        f"TM_rate={r.tm_rate_mm_per_s:.1f} mm/s  "
        f"clean_dur={r.clean_duration_tm_s:.1f}s"
    )

# Per-endpoint breakdown
tm_rows = []
for rid, res in tm_results.items():
    for ep in TM_ENDPOINTS:
        tm_rows.append({"run_id": rid, "endpoint": ep, "tm_mm": res.tm_per_endpoint[ep], "steps": res.step_counts[ep]})
tm_detail = pd.DataFrame(tm_rows)
display(tm_detail.pivot_table(index="endpoint", columns="run_id", values="tm_mm", aggfunc="first").round(0))

  671_T1_P2_R1_Take 2026-01-06 03.57.12 PM_002: TM=0.44 km  TM_rate=4140.5 mm/s  clean_dur=106.9s
  671_T2_P2_R1_Take 2026-01-15 04.35.25 PM_006: TM=0.46 km  TM_rate=4343.9 mm/s  clean_dur=106.3s
  671_T3_P2_R1_Take 2026-02-03 08.05.01 PM_001: TM=0.45 km  TM_rate=4218.8 mm/s  clean_dur=105.8s


run_id,671_T1_P2_R1_Take 2026-01-06 03.57.12 PM_002,671_T2_P2_R1_Take 2026-01-15 04.35.25 PM_006,671_T3_P2_R1_Take 2026-02-03 08.05.01 PM_001
endpoint,,,
LeftFoot,54459.0,65704.0,71583.0
LeftHand,165579.0,167082.0,158326.0
RightFoot,63950.0,72677.0,75545.0
RightHand,158496.0,156392.0,141059.0


---
## Block 3 — Shared PCA Engine

Reference-anchored PCA on 19 `{Joint}__zeroed_rel_omega_mag` columns (dynamics branch).  
StandardScaler + PCA fit on T1 (reference) only; all sessions projected into T1 basis.

In [ ]:
# ── Block 2.5: PCA & Feature Toggles ──────────────────────────────────
# Adjust optional analyses before running the PCA engine and downstream features.

w_session_native_gini = widgets.Checkbox(
    value=PARAMS_PCA_F4_F5.get("run_session_native_gini", True),
    description="Session-native Gini  (fit PCA per session for F5 comparison)",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="550px"),
)
w_session_native_deff = widgets.Checkbox(
    value=PARAMS_PCA_F4_F5.get("run_session_native_deff", True),
    description="Session-native D_eff  (fit PCA per session for F4 comparison)",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="550px"),
)
w_compute_ap_ratio = widgets.Checkbox(
    value=PARAMS_PCA_F4_F5.get("compute_ap_ratio", False),
    description="Axial / Peripheral ratio  (F5.1 — requires PCA joint attribution)",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="550px"),
)
w_kinematic_branch = widgets.Dropdown(
    options=[("dynamics (angular velocity)", "dynamics"),
             ("kinematics (position)", "kinematics")],
    value=PARAMS_PCA_F4_F5.get("kinematic_branch", "dynamics"),
    description="PCA branch:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="400px"),
)
w_n_components = widgets.IntSlider(
    value=PARAMS_PCA_F4_F5.get("n_components", 19),
    min=2, max=19, step=1,
    description="PCA components:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="400px"),
)

w_pca_apply_btn = widgets.Button(
    description="  Apply PCA Settings",
    button_style="info",
    icon="cogs",
    layout=widgets.Layout(width="240px", height="36px"),
)
w_pca_status = widgets.Output()


def _on_pca_apply(b):
    PARAMS_PCA_F4_F5["run_session_native_gini"] = w_session_native_gini.value
    PARAMS_PCA_F4_F5["run_session_native_deff"] = w_session_native_deff.value
    PARAMS_PCA_F4_F5["compute_ap_ratio"] = w_compute_ap_ratio.value
    PARAMS_PCA_F4_F5["kinematic_branch"] = w_kinematic_branch.value
    PARAMS_PCA_F4_F5["n_components"] = w_n_components.value

    with w_pca_status:
        clear_output(wait=True)
        checks = []
        if w_session_native_gini.value:
            checks.append("session-native Gini")
        if w_session_native_deff.value:
            checks.append("session-native D_eff")
        if w_compute_ap_ratio.value:
            checks.append("A/P ratio")
        summary = ", ".join(checks) if checks else "none"
        display(HTML(
            f"<b>PCA settings updated.</b> "
            f"Branch = <code>{w_kinematic_branch.value}</code>, "
            f"n_components = {w_n_components.value}, "
            f"optional features: {summary}.<br>"
            f"<i>Now run Block 3 and Block 4 below.</i>"
        ))


w_pca_apply_btn.on_click(_on_pca_apply)

display(widgets.VBox([
    widgets.HTML("<h4>PCA & Feature Settings</h4>"),
    widgets.HTML(
        "<p style='font-size:0.85em;color:#666'>"
        "Toggle optional analyses and PCA parameters. "
        "Click Apply, then run Block 3 (PCA) and Block 4 (F4/F5).</p>"
    ),
    w_kinematic_branch,
    w_n_components,
    widgets.HTML("<hr style='margin:6px 0'>"),
    widgets.HTML("<b>Optional features:</b>"),
    w_session_native_gini,
    w_session_native_deff,
    w_compute_ap_ratio,
    widgets.HTML("<hr style='margin:6px 0'>"),
    w_pca_apply_btn,
    w_pca_status,
]))

In [23]:
# ── Block 3: Shared PCA engine ────────────────────────────────────────

# Pre-PCA anchor checklist (§4.6)
ref_q = quality_df[quality_df["run_id"] == REFERENCE_RUN_ID].iloc[0]
print("=== Pre-PCA Anchor Checklist ===")
print(f"  Reference: {REFERENCE_RUN_ID}")
print(f"  artifact_frac:      {ref_q['global_artifact_frac']:.4f} (< 0.20? {'YES' if ref_q['global_artifact_frac'] < 0.20 else 'NO'})")
print(f"  clean_fraction_pca: {ref_q['clean_fraction_pca']:.4f} (>= 0.70? {'YES' if ref_q['clean_fraction_pca'] >= 0.70 else 'NO'})")
print(f"  clean_duration_s:   {ref_q['clean_duration_s']:.1f} (> 60s? {'YES' if ref_q['clean_duration_s'] > 60 else 'NO'})")
print()

pca_engine = build_pca_engine(active_sessions, REFERENCE_RUN_ID, PARAMS_PCA_F4_F5)

print(f"PCA engine built. Sessions transformed: {list(pca_engine.var_per_pc_by_run.keys())}")
print(f"Reference explained variance (first 5 PCs): {pca_engine.pca.explained_variance_ratio_[:5].round(4)}")
print(f"Cumulative explained variance (first 5 PCs): {np.cumsum(pca_engine.pca.explained_variance_ratio_[:5]).round(4)}")

for rid in active_run_ids:
    n_cl = pca_engine.n_clean_by_run.get(rid, 0)
    print(f"  {rid[:50]}: {n_cl} clean frames projected")

=== Pre-PCA Anchor Checklist ===
  Reference: 671_T1_P2_R1_Take 2026-01-06 03.57.12 PM_002
  artifact_frac:      0.0576 (< 0.20? YES)
  clean_fraction_pca: 0.8743 (>= 0.70? YES)
  clean_duration_s:   96.2 (> 60s? YES)

PCA engine built. Sessions transformed: ['671_T1_P2_R1_Take 2026-01-06 03.57.12 PM_002', '671_T2_P2_R1_Take 2026-01-15 04.35.25 PM_006', '671_T3_P2_R1_Take 2026-02-03 08.05.01 PM_001']
Reference explained variance (first 5 PCs): [0.1432 0.1324 0.0981 0.0735 0.0689]
Cumulative explained variance (first 5 PCs): [0.1432 0.2755 0.3737 0.4472 0.5161]
  671_T1_P2_R1_Take 2026-01-06 03.57.12 PM_002: 11541 clean frames projected
  671_T2_P2_R1_Take 2026-01-15 04.35.25 PM_006: 11228 clean frames projected
  671_T3_P2_R1_Take 2026-02-03 08.05.01 PM_001: 11309 clean frames projected


---
## Block 4 — F4: D_eff + F5: Joint Gini

**D_eff** (participation ratio): $D_{\text{eff}} = 1 / \sum p_k^2$ where $p_k = \lambda_k / \sum \lambda$  
**Joint Gini**: Gini on PCA-attributed per-joint variance proportions $\pi_j$  
Both read from the shared `pca_engine` — no second PCA fit for anchored mode.

In [24]:
# ── Block 4: F4 D_eff + F5 Joint Gini ─────────────────────────────────

deff_results: dict[str, object] = {}
gini_results: dict[str, object] = {}
ap_results: dict[str, dict] = {}

for rid in active_run_ids:
    deff_results[rid] = compute_d_eff(
        pca_engine, rid, PARAMS_PCA_F4_F5,
        session_df=active_sessions[rid],
    )
    gini_results[rid] = compute_joint_gini(
        pca_engine, rid, PARAMS_PCA_F4_F5,
        session_df=active_sessions[rid],
    )
    if PARAMS_PCA_F4_F5.get("compute_ap_ratio", False):
        ap_results[rid] = compute_ap_ratio(gini_results[rid].joint_proportions_anchored)
    
    dr = deff_results[rid]
    gr = gini_results[rid]
    native_deff_str = f"  native={dr.session_native_d_eff:.3f}" if dr.session_native_d_eff is not None else ""
    native_gini_str = f"  native={gr.gini_native:.4f}" if gr.gini_native is not None else ""
    print(
        f"  {rid[:50]}: D_eff={dr.d_eff:.3f} (norm={dr.d_eff_norm:.4f}, n90={dr.n90}){native_deff_str}  "
        f"Gini={gr.gini_anchored:.4f}{native_gini_str}"
    )

  671_T1_P2_R1_Take 2026-01-06 03.57.12 PM_002: D_eff=12.646 (norm=0.6656, n90=14)  native=12.646  Gini=0.0000  native=0.5562
  671_T2_P2_R1_Take 2026-01-15 04.35.25 PM_006: D_eff=11.624 (norm=0.6118, n90=14)  native=11.507  Gini=0.0393  native=0.4617
  671_T3_P2_R1_Take 2026-02-03 08.05.01 PM_001: D_eff=11.720 (norm=0.6168, n90=15)  native=11.122  Gini=0.1028  native=0.4326


---
## Block 5 — Summary & Export

Assemble `feature_scalars.csv` (one row per run_id) and `run_metadata.json`.

In [25]:
# ── Block 5: Summary table ─────────────────────────────────────────────

feature_rows = []
for rid in active_run_ids:
    row = assemble_feature_row(
        run_id=rid,
        atf_result=atf_results[rid],
        tm_result=tm_results[rid],
        deff_result=deff_results.get(rid),
        gini_result=gini_results.get(rid),
        ap_result=ap_results.get(rid),
    )
    reg = registry_df[registry_df["run_id"] == rid].iloc[0]
    row["subject_id"] = reg["subject_id"]
    row["timepoint"] = reg["timepoint"]
    row["protocol"] = reg["protocol"]
    row["repetition"] = reg["repetition"]
    feature_rows.append(row)

feature_scalars = pd.DataFrame(feature_rows)

# Reorder: identifiers first
id_cols = ["run_id", "subject_id", "timepoint", "protocol", "repetition"]
other_cols = [c for c in feature_scalars.columns if c not in id_cols]
feature_scalars = feature_scalars[id_cols + other_cols]

# Display summary
summary_cols = [
    "run_id", "timepoint", "repetition",
    "atf_wb", "atf_axial", "atf_peripheral",
    "tm_rate_mm_per_s", "d_eff", "d_eff_norm",
    "gini_anchored", "gini_native",
]
display(feature_scalars[summary_cols].style.format({
    "atf_wb": "{:.4f}", "atf_axial": "{:.4f}", "atf_peripheral": "{:.4f}",
    "tm_rate_mm_per_s": "{:.1f}", "d_eff": "{:.3f}", "d_eff_norm": "{:.4f}",
    "gini_anchored": "{:.4f}", "gini_native": "{:.4f}",
}))

,run_id,timepoint,repetition,atf_wb,atf_axial,atf_peripheral,tm_rate_mm_per_s,d_eff,d_eff_norm,gini_anchored,gini_native
0,671_T1_P2_R1_Take 2026-01-06 03.57.12 PM_002,T1,R1,0.9499,0.9499,0.9500,4140.5,12.646,0.6656,0.0000,0.5562
1,671_T2_P2_R1_Take 2026-01-15 04.35.25 PM_006,T2,R1,0.9499,0.9499,0.9500,4343.9,11.624,0.6118,0.0393,0.4617
2,671_T3_P2_R1_Take 2026-02-03 08.05.01 PM_001,T3,R1,0.9499,0.9499,0.9500,4218.8,11.720,0.6168,0.1028,0.4326


In [26]:
# ── Block 5.1: Write feature_scalars.csv ───────────────────────────────

csv_path = RESULTS_DIR / "feature_scalars.csv"
feature_scalars.to_csv(csv_path, index=False)
print(f"Wrote {csv_path} ({len(feature_scalars)} rows, {len(feature_scalars.columns)} cols)")

Wrote c:\Users\drorh\OneDrive - Mobileye\Desktop\gaga\results\meth_v2\feature_scalars.csv (3 rows, 43 cols)


In [27]:
# ── Block 5.2: Write run_metadata.json ─────────────────────────────────

quality_summary = quality_df[quality_df["run_id"].isin(active_run_ids)][
    ["run_id", "n_frames", "duration_s", "global_artifact_frac",
     "clean_fraction_pca", "hard_exclude", "soft_warning", "pca_low_confidence"]
].to_dict(orient="records")

run_metadata = {
    "spec_revision": "METHODOLOGY_SPEC_v2.md (v3.0)",
    "pipeline_version": "v2_feature_engine MVP",
    "timestamp_utc": datetime.datetime.utcnow().isoformat() + "Z",
    "subject_id": SUBJECT_ID,
    "reference_run_id": REFERENCE_RUN_ID,
    "active_run_ids": active_run_ids,
    "params": {
        "CONFIG": {k: v for k, v in CONFIG.items() if not isinstance(v, dict) or k == "time_windows"},
        "PARAMS_F1": PARAMS_F1,
        "PARAMS_F2": PARAMS_F2,
        "PARAMS_PCA_F4_F5": PARAMS_PCA_F4_F5,
    },
    "quality_summary": quality_summary,
    "bootstrap_run": False,
    "t2_isolation_run": False,
    "branch_d_active": False,
    "branch_d_implemented": False,
    "viz_exported": False,
    "session_native_gini_skipped": not PARAMS_PCA_F4_F5.get("run_session_native_gini", True),
    "session_native_deff_skipped": not PARAMS_PCA_F4_F5.get("run_session_native_deff", True),
}

meta_path = RESULTS_DIR / "run_metadata.json"
with open(meta_path, "w") as f:
    json.dump(run_metadata, f, indent=2, default=str)

print(f"Wrote {meta_path}")
print(f"\n=== MVP Export Complete ===")
print(f"  feature_scalars.csv : {csv_path}")
print(f"  run_metadata.json   : {meta_path}")

Wrote c:\Users\drorh\OneDrive - Mobileye\Desktop\gaga\results\meth_v2\run_metadata.json

=== MVP Export Complete ===
  feature_scalars.csv : c:\Users\drorh\OneDrive - Mobileye\Desktop\gaga\results\meth_v2\feature_scalars.csv
  run_metadata.json   : c:\Users\drorh\OneDrive - Mobileye\Desktop\gaga\results\meth_v2\run_metadata.json


---
## Block 6 — Optional Visualization (Deferred)

Deferred to v3.1. When implemented, call `v2_viz_engine.py` for Plotly HTML from tidy tables only.  
Must not refit PCA or embed raw kinematics.